In [1]:
from datetime import datetime

print(f"Timestamp: {datetime.now()}")

Timestamp: 2026-01-02 12:50:49.041743


# Prepare processed data for GEO upload

**Pinned Environment:** [`envs/sc-scvi.yaml`](../.../envs/sc-scvi.yaml)  

In [2]:
from pathlib import Path
import os
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import sparse
import warnings
import session_info
import sys

In [3]:
plt.rcParams['figure.figsize'] = (3, 3)
plt.rcParams['figure.dpi'] = 300

### Setup

In [4]:
sys.path.append(str(Path.cwd().resolve().parents[1]))

from config.paths import BASE_DIR

In [5]:
# Input
adata_dir = BASE_DIR / "data/h5ad/export_01/01b_filter"
adata_path = adata_dir / "adata-480.h5ad" # Unsliced adata.var

bdata_dir = BASE_DIR / "data/h5ad/export_03/03c_thresholds"
bdata_path = bdata_dir / "adata-labeled.h5ad" # Intermediate labels (all cell types)

cdata_dir = BASE_DIR / "data/h5ad/export_04/04c_gamma_subtypes"
cdata_path = cdata_dir / "neurons-final-labels.h5ad" # Neuron labels (neurons only)

# Output
output_dir = "/home/workspace/temp/geo_organized"
os.makedirs(output_dir, exist_ok=True)

In [6]:
adata = sc.read_h5ad(adata_path)
bdata = sc.read_h5ad(bdata_path)
cdata = sc.read_h5ad(cdata_path)

In [7]:
adata

AnnData object with n_obs × n_vars = 147036 × 480
    obs: 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'sample_id', 'output_id', 'region_id', 'slide_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'batch'
    obsm: 'spatial'
    layers: 'counts'

### Strip extra annotations in adata.obs

In [8]:
keep_obs = [
    'cell_id', 'x_centroid', 'y_centroid',
    'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 
    'sample_id', 'slide_id', 
    'total_counts', 'n_genes_by_counts']

keep_uns = ['neighbors', 'umap']
keep_obsm = ['X_scVI', 'X_umap', 'X_spatial']
keep_layers = ['counts', 'log1p']
keep_obsp = ['connectivities', 'distances']

In [9]:
# Clean .obs
for col in list(adata.obs.columns):
    if col not in keep_obs:
        del adata.obs[col]

# Clean .uns
for key in list(adata.uns.keys()):
    if key not in keep_uns:
        del adata.uns[key]

# Clean .obsm
for key in list(adata.obsm.keys()):
    if key not in keep_obsm:
        del adata.obsm[key]

# Clean .layers
for key in list(adata.layers.keys()):
    if key not in keep_layers:
        del adata.layers[key]

# Clean .obsp
for key in list(adata.obsp.keys()):
    if key not in keep_obsp:
        del adata.obsp[key]

In [10]:
adata.X = adata.layers["counts"].copy()

In [11]:
adata

AnnData object with n_obs × n_vars = 147036 × 480
    obs: 'cell_id', 'x_centroid', 'y_centroid', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'sample_id', 'slide_id', 'n_genes_by_counts'
    layers: 'counts'

## Copy intermediate labels

In [12]:
int_columns = ['seurat_labels', 'seurat_class', 'EGFP_Seq1_hi', 'dTomato_Seq2_hi', 'EGFP_dTomato_dual_hi']
adata.obs[int_columns] = bdata.obs[int_columns].values

for col in int_columns:
    adata.obs[col] = adata.obs[col].astype(str).astype('category') # helps with string export error
    
adata.obs[int_columns].head()

,seurat_labels,seurat_class,EGFP_Seq1_hi,dTomato_Seq2_hi,EGFP_dTomato_dual_hi
aaadagcf-1-0,Nonpeptidergic nociceptors,Neuron,False,False,False
aaaihgjh-1-0,Adelta-LTMR,Neuron,False,False,False
aabcohae-1-0,Adelta-LTMR,Neuron,False,False,False
aabdncpn-1-0,Proprioceptors,Neuron,False,False,False
aabkbaii-1-0,SGC,Non-neuron,False,False,False


## Map neuron annotations to adata

In [13]:
neuron_columns = ['neuron_labels', 'reporter_status', 'innervation_pattern']

# Create a mapping dictionary from cdata using 'cell_id' as the key since barcodes are different
for col in neuron_columns:
    mapping = dict(zip(cdata.obs['cell_id'], cdata.obs[col]))
    adata.obs[col] = adata.obs['cell_id'].map(mapping)

adata.obs[neuron_columns].head()

,neuron_labels,reporter_status,innervation_pattern
aaadagcf-1-0,Nonpeptidergic nociceptors,Negative,Negative
aaaihgjh-1-0,Adelta-LTMR,Negative,Negative
aabcohae-1-0,Adelta-LTMR,Negative,Negative
aabdncpn-1-0,Proprioceptors,Negative,Negative
aabkbaii-1-0,NaN,NaN,NaN


## Fix column names and formatting

In [14]:
adata.obs['neuron_filter'] = 'NA' # Default for non-neurons
neuron_mask = adata.obs['seurat_class'] == 'Neuron'
passed_qc = adata.obs['cell_id'].isin(cdata.obs['cell_id'])

adata.obs.loc[neuron_mask & passed_qc, 'neuron_filter'] = 'Pass'
adata.obs.loc[neuron_mask & ~passed_qc, 'neuron_filter'] = 'Fail'

In [15]:
# rename some columns for clarity since we end up using seurat and not scanvi
rename_dict = {
    'seurat_labels': 'cell_type',
    'seurat_class': 'cell_class'
}

adata.obs.rename(columns=rename_dict, inplace=True)

In [16]:
# Add the CGRP-Gamma subtypes from cdata to adata (for only neurons, use cell_class annotation)
adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)
adata.obs['cell_class'] = adata.obs['cell_class'].astype(str)

neuron_mapping = dict(zip(cdata.obs['cell_id'], cdata.obs['neuron_labels']))
neuron_mask = adata.obs['cell_class'] == 'Neuron'
adata.obs.loc[neuron_mask, 'cell_type'] = adata.obs.loc[neuron_mask, 'cell_id'].map(neuron_mapping)

In [17]:
# Remove the redundant column
adata.obs.drop(columns=['neuron_labels'], inplace=True, errors='ignore')
adata.obs = adata.obs.apply(lambda x: x.cat.remove_unused_categories() if x.dtype == 'category' else x)

In [18]:
categories = ['cell_type', 'cell_class']

for cat in categories:
    adata.obs[cat] = adata.obs[cat].astype("category")
    adata.obs[cat] = adata.obs[cat].cat.remove_unused_categories()

In [19]:
# Update label colors and assign to adata.uns
cell_types = adata.obs["cell_type"].cat.categories
colors = list(sns.color_palette("husl", n_colors=len(cell_types)).as_hex())
adata.uns["cell_type_colors"] = colors

# Add multi-dimensional observations

### Latent rep

In [20]:
# Create placeholder in the full data object
n_latent = cdata.obsm['X_scVI_seurat_neuron'].shape[1]
full_scvi = np.full((adata.n_obs, n_latent), np.nan)

# Define the map
neuron_idx_map = dict(zip(cdata.obs['cell_id'], range(len(cdata))))

# Map coordinates by cell_id
for i, cid in enumerate(adata.obs['cell_id']):
    if cid in neuron_idx_map:
        full_scvi[i] = cdata.obsm['X_scVI_seurat_neuron'][neuron_idx_map[cid]]

# Assign the mapped matrix to the full object
adata.obsm['X_scVI'] = full_scvi

# verify
passed = adata.obs['neuron_filter'] == 'Pass'
mapped_scvi = ~np.isnan(adata.obsm['X_scVI'][:, 0])
print(f"Total cells in full adata: {len(adata)}")
print(f"Neurons flagged as 'Pass': {passed.sum()}")
print(f"Rows with scVI latent data: {mapped_scvi.sum()}")
print(f"Check - Every 'Pass' cell has scVI: {(mapped_scvi[passed]).all()}")
print(f"Check - No 'Fail' cells have scVI: {(~mapped_scvi[~passed]).all()}")

Total cells in full adata: 147036
Neurons flagged as 'Pass': 22411
Rows with scVI latent data: 22411
Check - Every 'Pass' cell has scVI: True
Check - No 'Fail' cells have scVI: True


### UMAP

In [21]:
full_umap = np.full((adata.n_obs, 2), np.nan)

neuron_idx_map = dict(zip(cdata.obs['cell_id'], range(len(cdata))))
for i, cid in enumerate(adata.obs['cell_id']):
    if cid in neuron_idx_map:
        full_umap[i] = cdata.obsm['X_umap'][neuron_idx_map[cid]]

adata.obsm['X_umap'] = full_umap

is_neuron = adata.obs['cell_class'] == 'Neuron'
neuron_umap_data = adata.obsm['X_umap'][is_neuron]
non_neuron_umap_data = adata.obsm['X_umap'][~is_neuron]
print(f"Total neurons in metadata: {is_neuron.sum()}")
print(f"Neurons successfully mapped: {np.count_nonzero(~np.isnan(neuron_umap_data[:, 0]))}")
print(f"Actual neurons missing coordinates: {np.isnan(neuron_umap_data).any(axis=1).sum()}") 
print(f"Non-neurons are correctly NaN: {np.isnan(non_neuron_umap_data).all()}")

Total neurons in metadata: 63591
Neurons successfully mapped: 22411
Actual neurons missing coordinates: 41180
Non-neurons are correctly NaN: True


# Export

In [22]:
adata.write(os.path.join(output_dir, 'adata_processed.h5ad'), compression = 'gzip')

# Session info

In [23]:
print('active conda environment:', os.path.basename(sys.prefix))
session_info.show()

active conda environment: sc-charter


/home/workspace/environment/sc-charter/lib/python3.12/site-packages/session_info/main.py:213: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  mod_version = _find_version(mod.__version__)
